In [1]:
from Bio import SeqIO
from Bio import SeqUtils
from Bio.Seq import Seq
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os
import numpy as np
import pandas as pd
pd.set_option("display.float_format",lambda x:'%.2f'%x) # 取消科学计数显示
import time
print (time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()) )
import warnings
warnings.filterwarnings("ignore")
from collections import Counter
from matplotlib import pyplot as plt
import seaborn as sns
table_path='C:\\Users\\huangyan8\\Desktop\\work\\2024-10-22 TE_Evolution_Draft\\Tables\\'
fig_path='C:\\Users\\huangyan8\\Desktop\\work\\2024-10-22 TE_Evolution_Draft\\Figures\\'

2024-12-31 12:40:29


In [26]:
def get_kmers(x):
    if x=='':
        return ''
    tepeps=x.split(", ")
    kmers=[]
    for tepep in tepeps[:1000]:
        if tepep in fa_dic:
            seq=fa_dic[tepep]
            for i in range(len(seq)-7):
                kmer=seq[i:i+7]
                if 'N' not in kmer:
                    kmers.append(kmer)
    return " ".join(kmers)+" "
def get_query_kmers(x):
    if x=='':
        return ''
    kmers=[]
    for i in range(len(x)-7):
        kmer=x[i:i+7]
        if 'N' not in kmer:
            kmers.append(kmer)
    return " ".join(kmers)

In [5]:
tax_path='D:\\18.TE_Evolution\\00.Sample_Information\\Genomes_Info\\taxdmp\\'
tax=pd.read_csv(tax_path+"nodes.dmp",sep='\t').fillna("")
use_col=[]

for col in tax.columns:
    #print(col,tax.loc[0,col])
    if tax.loc[0,col]!="|":
        use_col.append(col)
tax=tax[use_col]
tax.columns=["tax_id",'parent tax_id','rank','embl code','division id','inherited div flag','genetic code id','inherited GC flag',
            'mitochondrial genetic code id','inherited MGC flag','GenBank hidden flag','hidden subtree root flag','comments']
tax.head(1)

,tax_id,parent tax_id,rank,embl code,division id,inherited div flag,genetic code id,inherited GC flag,mitochondrial genetic code id,inherited MGC flag,GenBank hidden flag,hidden subtree root flag,comments
0,2,131567,superkingdom,,0,0,11,0,0,0,0,0,


In [2]:
table_path='C:\\Users\\huangyan8\\Desktop\\work\\2024-12-03 TE HT Draft\\Tables\\'
samples=pd.read_excel(table_path+"Table S1\\Table S1.Sample_Information.xlsx").fillna("")
taxid_dic={}
for i in range(samples.shape[0]):
    taxid_dic[samples.loc[i,"Specie_ID"]]=samples.loc[i,'Taxonomy_ID']

### Step 1. Species-wise grouping

In [3]:
table_path='C:\\Users\\huangyan8\\Desktop\\work\\2024-12-03 TE HT Draft\\Tables\\'
species_grouping=pd.read_excel(table_path+"Table S1\\Table S1.Sample_Grouping.xlsx").fillna("")
species_grouping.head(1)

,Taxonomy_ID,Species_Name,Species_samples,Closely_related_species_samples,Notes,Distantly_related_species_samples
0,2711,Citrus sinensis,XBSZ,PBXX | QIBM | VMRD | DXPQ | LFYN | NAPF | SHMU...,,BUKK | GJNO | WWFJ | ZVZR | XNAW | XHDS | VMNX...


In [4]:
sample_grouping_dic={}
for i in range(species_grouping.shape[0]):
    for sample in species_grouping.loc[i,'Species_samples'].split(" | "):
        sample_grouping_dic[sample]={}
        sample_grouping_dic[sample]['Closely_related_species_samples']=species_grouping.loc[i,'Closely_related_species_samples'].split(" | ")
        sample_grouping_dic[sample]['Distantly_related_species_samples']=species_grouping.loc[i,'Distantly_related_species_samples'].split(" | ")
        
for k in sample_grouping_dic['XBSZ'].keys():
    print(k,len(sample_grouping_dic['XBSZ'][k]))

Closely_related_species_samples 12
Distantly_related_species_samples 545


### Step 2. Screen TE families with more than one homologous sample

In [57]:
for family in  ['CACTA', 'Copia', 'Gypsy', 'hAT', 'Helitron', 'Mariner', 'Mutator', 'PIF-Harbinger', 'Other']:#['Other' 'Gypsy','Copia','PIF-Harbinger','CACTA', 'Mutator' ,'Helitron','hAT','LTR_Roo','LINE' ,'Mariner']:
    ortho_path='D:\\19.TE_HT\\03.TE_Ortho\\'+family+'\\Orthogroups\\'
    ortho=pd.read_csv(ortho_path+"Orthogroups_V5.tsv",sep='\t').fillna("")
    Group=[]
    Samples=[]
    ortho=ortho.set_index("Orthogroup")
    print(family,ortho.shape)
    #ortho=ortho.set_index("Orthogroup")
    for group in ortho.index:
        df0=pd.DataFrame(ortho.loc[group,:])
        df0=df0[(df0[group]!='')&(df0[group]!=' ')]
        Group.append(group)
        Samples.append(" | ".join(list(df0.index)))
    data=pd.DataFrame(Group,columns=['Group'])
    data['Group_Samples']=pd.DataFrame(Samples)
    data['Sample_Count']=data['Group_Samples'].apply(lambda x:len(x.split(" | ")))
    print(family,ortho.shape[0],data[data['Sample_Count']>1].shape[0])
    data.to_csv(ortho_path+"Orthogroups_Group_Samples.tsv",sep='\t',index=False)

CACTA (14922, 558)
CACTA 14922 9381
Copia (30109, 548)
Copia 30109 20249
Gypsy (28264, 550)
Gypsy 28264 16625
hAT (17740, 555)
hAT 17740 11451
Helitron (10261, 558)
Helitron 10261 7088
Mariner (14886, 548)
Mariner 14886 10991
Mutator (29871, 558)
Mutator 29871 18773
PIF-Harbinger (9233, 547)
PIF-Harbinger 9233 6031
Other (24988, 555)
Other 24988 15270


### Step 3. Screen HTT Species Pairs——Reciept Species
- 当前species有，但上溯2级祖先的旁系同源物种，没有TE domain family;为候选Target speices

In [5]:
def get_sample_type(group_samples):
    group_samples=group_samples.split(" | ")
    HT_targets=[]
    for sample in group_samples:
        ht=True
        if sample in sample_grouping_dic:
            if 'Closely_related_species_samples' in sample_grouping_dic[sample]:
                Closely_related_species_samples=sample_grouping_dic[sample]['Closely_related_species_samples']
                for query in Closely_related_species_samples:
                    if query in group_samples:
                        ht=False
                if ht==True:
                    HT_targets.append(sample)
    return " | ".join(HT_targets)

In [72]:
for family in  ['CACTA', 'Copia', 'Gypsy', 'hAT', 'Helitron', 'Mariner', 'Mutator', 'PIF-Harbinger', 'Other']:
    ortho_path='D:\\19.TE_HT\\03.TE_Ortho\\'+family+'\\Orthogroups\\'
    data=pd.read_csv(ortho_path+"Orthogroups_Group_Samples.tsv",sep='\t')
    data=data[data['Group_Samples'].apply(lambda x:1 if type(x)!=str else 0)==0]
    data['HTT']=data['Group_Samples'].apply(get_sample_type)
    print(family,data[(data['HTT']!='')].shape)
    data.to_csv(ortho_path+"Orthogroups_Group_Samples.tsv",sep='\t',index=False)

CACTA (7874, 4)
Copia (16968, 4)
Gypsy (17204, 4)
hAT (9435, 4)
Helitron (5662, 4)
Mariner (5350, 4)
Mutator (14886, 4)
PIF-Harbinger (4291, 4)
Other (13664, 4)


In [73]:
for family in  ['CACTA', 'Copia', 'Gypsy', 'hAT', 'Helitron', 'LINE', 'LTR_Roo', 'Mariner', 'Mutator', 'Other']:
    ortho_path='D:\\19.TE_HT\\01.TEpep_Ortho\\'+family+'\\Orthogroups\\'
    data=pd.read_csv(ortho_path+"Orthogroups_Group_Samples.tsv",sep='\t')
    data=data[data['Sample_Count']>1]
    data=data[data['Group_Samples'].apply(lambda x:1 if type(x)!=str else 0)==0]
    data['HTT']=data['Group_Samples'].apply(get_sample_type)
    print(family,data[(data['HTT']!='')].shape)
    data.to_csv(ortho_path+"Orthogroups_Group_Samples.tsv",sep='\t',index=False)

CACTA (13438, 6)
Copia (167539, 6)
Gypsy (49923, 6)
hAT (5582, 6)
Helitron (5820, 6)
LINE (46219, 6)
LTR_Roo (844, 6)
Mariner (128, 6)
Mutator (9371, 6)
Other (920, 6)


### Step 4. The distantly related species with the highest similarity——Donor Speices
- 针对每个HTT 的Target species，寻找TE domain序列相似度最高的Species（非同一genus,进化谱系至少差两个等级）

In [6]:
samples=pd.read_excel('../data/Table S1.TE_Evo_558_Species.xlsx')
taxid_dic={}
for i in range(samples.shape[0]):
    taxid_dic[samples.loc[i,"Specie_ID"]]=samples.loc[i,'Name']#

In [ ]:
Family=[]
Group=[]
HT_Target=[]
HT_Source=[]
HT_Sim=[]
for family in  ['hAT', 'Helitron', 'Mariner', 'Mutator', 'PIF-Harbinger', 'Other','CACTA', 'Copia', 'Gypsy']:
    ortho_path='D:\\19.TE_HT\\03.TE_Ortho\\'+family+'\\Orthogroups\\'
    data=pd.read_csv(ortho_path+"Orthogroups_Group_Samples.tsv",sep='\t').fillna('')
    #data=data[(data['HTT']!='')&(data['VTT']!="")].reset_index(drop=True)
    data=data[(data['HTT']!='')].reset_index(drop=True)
    use={}
    for g in data['Group'].tolist():
        use[g]=1
    print(family,data.shape)
    ortho=pd.read_csv(ortho_path+"Orthogroups_V5.tsv",sep='\t').fillna("")
    ortho=ortho[ortho['Orthogroup'].apply(lambda x:1 if x in use else 0)==1].reset_index(drop=True)
    for col in ortho.columns[1:-1]:
        if col.split(".")[0] not in taxid_dic:
            del ortho[col]
    print(ortho.shape)
    
    seq_path='D:\\19.TE_HT\\00.Sequence\\TE\\'+family+"\\"
    for col in ortho.columns[1:-1]:
        fa_dic={}
        if col+".TE.fa" in os.listdir(seq_path):
            for s in SeqIO.parse(seq_path+col+".TE.fa",'fasta'):
                fa_dic[s.id]=str(s.seq)
        ortho[col]=ortho[col].apply(get_kmers)
        
    ortho=ortho.set_index("Orthogroup").T
    count=0
    for g in ortho.columns:
        count+=1
        if count%1000==0:
            print(count)
        df=ortho[[g]]
        df=df[(df[g]!='')&(df[g]!=' ')]
        if df.shape[0]>0:
            features=list(df.index)
            sequences=df[g].tolist()
            vectorizer = TfidfVectorizer(max_features=10000)
            tfidf_matrix = vectorizer.fit_transform(sequences)
            #print(tfidf_matrix.shape)
            sim_matrix=pd.DataFrame(cosine_similarity(tfidf_matrix,tfidf_matrix))
            sim_matrix.columns=features
            sim_matrix.index=features
            HTT_samples=data[data['Group']==g]['HTT'].tolist()[0].split(" | ")
            for s in HTT_samples:
                if s in sim_matrix.columns:
                    df0=sim_matrix[[s]]
                    df0=df0.sort_values(s,ascending=False).reset_index()
                    if df0.shape[0]>0:
                        for source_idx in range(df0.shape[0]):
                            source=df0.loc[source_idx,"index"]
                            if source in sample_grouping_dic[s]['Distantly_related_species_samples']:
                                Family.append(family)
                                Group.append(g)
                                HT_Target.append(s)
                                HT_Source.append(source)
                                HT_Sim.append(df0.loc[source_idx,s])
            del sim_matrix
    print(family,len(Family))
R=pd.DataFrame(Family,columns=['Family'])
R['Group']=pd.DataFrame(Group)
R['HT_Target']=pd.DataFrame(HT_Target)
R['HT_Source']=pd.DataFrame(HT_Source)
R['HT_Sim']=pd.DataFrame(HT_Sim)
R.head()

In [13]:
samples=pd.read_excel(table_path+"Table S1\\Table S1.Sample_Information.xlsx")
taxid_dic2={}
for i in range(samples.shape[0]):
    taxid_dic2[samples.loc[i,"Specie_ID"]]=samples.loc[i,'Name']#
R['Target_Species']=R['HT_Target'].apply(lambda x:taxid_dic2[x] if x in taxid_dic2 else '')
R['Source_Species']=R['HT_Source'].apply(lambda x:taxid_dic2[x] if x in taxid_dic2 else '')
#R[R['Target_Species']==R['Source_Species']]
print(R.shape)
R.to_csv("D:\\19.TE_HT\\Work\\HTT_TE_Species_Pairs.V3.csv",index=False)

(3638350, 7)


In [15]:
df1=R[(R["HT_Sim"]>0.8)&(R['Source_Species']!='')]
print(df1.shape)
df1=df1.sort_values("HT_Sim",ascending=False).reset_index(drop=True)
df1['Target_Genus']=df1['Target_Species'].apply(lambda x:x.split(" ")[0])
df1['Source_Genus']=df1['Source_Species'].apply(lambda x:x.split(" ")[0])
df1=df1[df1['Target_Genus']!=df1['Source_Genus']]
del df1['Target_Genus']
del df1['Source_Genus']
df1.to_excel("D:\\19.TE_HT\\Work\\HTT_TE_Species_Pairs_Pos_Stats.V3.xlsx",index=False)
df1['Group'].value_counts()

Group
TE_Helitron_OG0000010    2338
TE_Gypsy_OG0021796       1956
TE_Mutator_OG0000004     1593
TE_Helitron_OG0000043    1293
TE_Helitron_OG0000037    1203
                         ... 
TE_Copia_OG0001972          1
TE_Helitron_OG0000402       1
TE_CACTA_OG0000032          1
TE_Helitron_OG0000098       1
TE_Gypsy_OG0000221          1
Name: count, Length: 622, dtype: int64

In [16]:
len(list(df1['Group'].unique()))

622